# 8. HMM (Hidden Markov Model, 은닉 마르코프 모형)

드디어 레짐을 직접 다루는 개념이야. 순서는 이래.

1. 레짐을 확률모형으로 표현하기
2. 마르코프 체인과 전이행렬
3. 레짐별 수익률 분포
4. 오늘 레짐 추정하기(필터링)와 숫자 예시
5. 스무딩과 look-ahead 문제
6. 파라미터 추정(EM)
7. 예측
8. 함정과 코드

---

## 0단계: 핵심 아이디어

시장에 "평온장"과 "혼란장" 같은 **숨겨진 상태**가 있다고 가정하자. 우리는 그 상태를 **직접 볼 수 없고**, 매일의 수익률만 관측해. 하지만 상태에 따라 수익률의 분포가 다르니, 관측된 수익률을 보고 **"지금 어느 상태일 확률이 얼마인가"를 거꾸로 추론**할 수 있어.

이름을 풀면 이래.

- **Hidden**: 상태(레짐)는 숨겨져 있다.
- **Markov**: 상태는 마르코프 성질을 따라 바뀐다(1단계).
- **Model**: 이 둘을 결합한 확률모형이다.

---

## 1단계: 마르코프 체인 — 레짐이 바뀌는 규칙

t일의 레짐을 $S_t$라 하자. 두 개 레짐이면 $S_t \in \{1, 2\}$(1 = 평온, 2 = 혼란)야.

**마르코프 성질:** 내일 레짐은 **오늘 레짐에만** 의존하고, 그 이전 역사는 상관없다.

$$
P(S_t = j \mid S_{t-1}, S_{t-2}, \dots) = P(S_t = j \mid S_{t-1})
$$

이 확률들을 **전이행렬(transition matrix)**로 정리해.

$$
P = \begin{pmatrix} p_{11} & p_{12} \\ p_{21} & p_{22} \end{pmatrix}, \qquad p_{ij} = P(S_t = j \mid S_{t-1} = i)
$$

각 **행의 합은 1**이야. 오늘 레짐 i에서 출발하면 내일은 어딘가로 반드시 가니까. 예를 들어:

$$
P = \begin{pmatrix} 0.98 & 0.02 \\ 0.05 & 0.95 \end{pmatrix}
$$

- 평온장이면 내일도 평온할 확률은 98%, 혼란으로 넘어갈 확률은 2%야.
- 혼란장이면 내일도 혼란일 확률은 95%, 평온으로 돌아갈 확률은 5%야.

대각선 값이 1에 가까울수록 레짐이 **오래 지속**돼.

**기대 지속기간 (유도):** 레짐 i에 머무는 날수 D는 "매일 확률 $1-p_{ii}$로 탈출하는" 기하분포를 따라.

$$
P(D = d) = p_{ii}^{\,d-1}(1-p_{ii}) \quad\Rightarrow\quad E[D] = \frac{1}{1-p_{ii}}
$$

- 평온장: $1/0.02 = 50$일
- 혼란장: $1/0.05 = 20$일

**장기적으로 각 레짐에 있는 비율 (정상분포):** 장기 비율 $\pi_1, \pi_2$는 "레짐 1로 들어오는 흐름 = 나가는 흐름"이라는 균형 조건으로 구할 수 있어.

$$
\pi_1 \, p_{12} = \pi_2 \, p_{21}, \quad \pi_1 + \pi_2 = 1 \quad\Rightarrow\quad \pi_1 = \frac{p_{21}}{p_{12}+p_{21}} = \frac{0.05}{0.07} \approx 71\%
$$

즉 전체 기간의 약 71%는 평온장, 29%는 혼란장이야.

---

## 2단계: 방출분포 — 레짐별로 수익률이 어떻게 생겼나

각 레짐에서 수익률은 서로 다른 정규분포를 따른다고 가정해. 이걸 **방출분포(emission distribution)**라고 불러.

$$
r_t \mid S_t = k \;\sim\; N(\mu_k,\ \sigma_k^2)
$$

예시 값을 이렇게 두자(일별, %).

| 레짐 | 평균 $\mu_k$ | 변동성 $\sigma_k$ |
|---|---|---|
| 1 평온 | +0.05 | 0.8 |
| 2 혼란 | −0.10 | 2.5 |

정규분포 밀도함수는 앞으로 계속 쓰니 적어둘게.

$$
f_k(r) = \frac{1}{\sigma_k\sqrt{2\pi}}\exp\left(-\frac{(r-\mu_k)^2}{2\sigma_k^2}\right)
$$

$f_k(r)$은 "레짐 k였다면 수익률 r이 나올 **가능도(likelihood)**"야. 값이 클수록 그 레짐에서 자연스러운 관측이라는 뜻이지.

**HMM의 모든 파라미터는 이게 전부야:** 전이확률($p_{ij}$), 레짐별 평균($\mu_k$), 레짐별 변동성($\sigma_k$).

---

## 3단계: 필터링 — 오늘까지의 데이터로 오늘 레짐 추정하기 (핵심 유도)

목표는 이 확률이야.

$$
\xi_t(k) = P(S_t = k \mid r_1, \dots, r_t)
$$

**오늘까지의 데이터만** 보고 판단한 "오늘 레짐 k일 확률"이야. 이걸 **필터링 확률(filtered probability)**이라고 하고, 매일 두 스텝을 반복해서 구해(forward algorithm, Hamilton filter).

**스텝 ① 예측(Predict):** 어제의 판단을 전이행렬로 하루 굴려.

$$
P(S_t = j \mid r_1,\dots,r_{t-1}) = \sum_{i} \xi_{t-1}(i)\, p_{ij}
$$

"어제 i였을 확률 × i에서 j로 갈 확률"을 모든 i에 대해 합한 거야.

**스텝 ② 갱신(Update):** 오늘 수익률 $r_t$를 보고 베이즈 정리로 고쳐.

$$
\xi_t(j) = \frac{P(S_t = j \mid r_{1:t-1})\cdot f_j(r_t)}{\sum_k P(S_t = k \mid r_{1:t-1})\cdot f_k(r_t)}
$$

분자는 **사전 믿음 × 가능도**, 분모는 확률 합을 1로 맞추는 정규화야. 베이즈 정리 "사후 ∝ 사전 × 가능도" 그대로지.

---

## 4단계: 숫자로 필터 돌려보기

어제의 판단이 평온 90%, 혼란 10%였다고 하자: $\xi_{t-1} = (0.9,\ 0.1)$.

**오늘 수익률 $r_t = -3\%$가 관측됐다.**

**① 예측**

- $P(\text{평온}) = 0.9 \times 0.98 + 0.1 \times 0.05 = 0.887$
- $P(\text{혼란}) = 0.9 \times 0.02 + 0.1 \times 0.95 = 0.113$

**② 가능도**

- 평온: $(-3 - 0.05)/0.8 = -3.81$ → $f_1 \approx 0.4987 \times e^{-7.27} \approx 0.00035$
- 혼란: $(-3 + 0.10)/2.5 = -1.16$ → $f_2 \approx 0.1596 \times e^{-0.67} \approx 0.0814$

평온장에서 −3%는 거의 불가능한 값(3.8 표준편차)이지만, 혼란장에서는 흔한 값(1.2 표준편차)이야.

**③ 갱신**

- 평온: $0.887 \times 0.00035 = 0.00031$
- 혼란: $0.113 \times 0.0814 = 0.00920$
- 합: $0.00951$

$$
\xi_t = (0.00031/0.00951,\ 0.00920/0.00951) = (\mathbf{3\%},\ \mathbf{97\%})
$$

**−3% 하루 만에 판단이 "평온 90%"에서 "혼란 97%"로 뒤집혔어.**

**다음 날 $r_{t+1} = +0.5\%$가 관측됐다.**

- 예측: 평온 $= 0.03 \times 0.98 + 0.97 \times 0.05 \approx 0.080$, 혼란 $\approx 0.920$
- 가능도: $f_1 \approx 0.426$, $f_2 \approx 0.155$ (평범한 +0.5%는 평온장에서 더 자연스러워)
- 갱신: 평온 $0.080 \times 0.426 = 0.034$, 혼란 $0.920 \times 0.155 = 0.143$

$$
\xi_{t+1} \approx (\mathbf{19\%},\ \mathbf{81\%})
$$

**두 가지를 관찰할 수 있어.**

1. **평범한 날이 와도 바로 평온으로 돌아가지 않아.** 전이행렬의 $p_{22} = 0.95$(혼란은 오래 간다)가 판단에 **관성**을 줘. 평온장다운 날이 며칠 더 쌓여야 되돌아가.
2. **반대로, 극단적인 하루는 판단을 순식간에 뒤집어.** 단 하루의 −3%가 "혼란장 진입"으로 해석될 수 있다는 거야. 그 −3%가 그냥 일회성 악재였다면 **가짜 경보(false alarm)**가 되는 거지. 이게 10번 statistical jump model이 등장한 이유야. 미리 기억해둬.

---

## 5단계: 스무딩 — 그리고 치명적인 look-ahead 문제

필터링은 **오늘까지**의 데이터만 써. 그런데 과거 분석을 할 때는 **전체 표본(미래 포함)**을 보고 레짐을 판단할 수도 있어.

$$
\gamma_t(k) = P(S_t = k \mid r_1, \dots, r_T), \quad T > t
$$

이걸 **스무딩 확률(smoothed probability)**이라고 해. 앞의 필터를 끝까지 돌린 뒤, 끝에서부터 **거꾸로 한 번 더** 계산해서 구해(forward-backward algorithm).

4단계 예시로 보면, 필터는 −3% 당일에 혼란 97%라고 판단했어. 그런데 **그다음 20일이 모두 평온했다는 걸 알고 나면**, 스무딩은 "그날은 그냥 평온장의 이상치였네"라고 판단을 **사후적으로 수정**해.

**스무딩 확률은 레짐을 "그림으로 보여주기"에는 좋지만, 예측이나 백테스트에 쓰면 미래 정보를 쓴 거야.** "그날 혼란장이었으니 매도했다"는 전략을 스무딩 확률로 백테스트하면, 사실은 **미래를 보고** 과거 레짐을 판단한 거라 성과가 엄청나게 부풀려져. 처음 대화에서 말한 look-ahead bias가 정확히 이거야. 16번 "online 적용"에서 이 문제를 본격적으로 다룰 거야.

**비터비(Viterbi) 알고리즘**이라는 것도 있어. 각 날짜별 확률이 아니라 "전체 기간에서 가장 그럴듯한 레짐 경로 하나"를 찾아줘. 이것도 **전체 표본을 쓰기 때문에** 스무딩과 같은 문제가 있어.

---

## 6단계: 파라미터 추정 — EM 알고리즘 (Baum-Welch)

지금까지는 파라미터($p_{ij}$, $\mu_k$, $\sigma_k$)를 안다고 가정했어. 실제로는 데이터로 추정해야 하는데, **닭과 달걀 문제**가 있어.

- 레짐을 알면 → 레짐별 평균과 분산을 계산하기 쉬워.
- 파라미터를 알면 → 레짐 확률을 계산하기 쉬워(3~5단계).

그래서 **번갈아 가며 반복**해. 이게 EM(Expectation-Maximization)이야.

**E-step:** 현재 파라미터로 각 날짜의 레짐 확률 $\gamma_t(k)$를 계산해(스무딩).

**M-step:** 그 확률을 **가중치**로 써서 파라미터를 갱신해. 식을 보면 직관적이야.

$$
\mu_k = \frac{\sum_t \gamma_t(k)\, r_t}{\sum_t \gamma_t(k)}, \qquad \sigma_k^2 = \frac{\sum_t \gamma_t(k)\,(r_t - \mu_k)^2}{\sum_t \gamma_t(k)}
$$

레짐 k의 평균은 "레짐 k였을 확률로 가중한 수익률 평균"이야. 레짐 k였을 확률이 90%인 날은 크게, 5%인 날은 작게 반영하는 거지. 전이확률도 비슷하게 "i에서 j로 넘어간 기대 횟수 ÷ i에 있었던 기대 횟수"로 갱신해.

이 두 스텝을 결과가 더 이상 안 바뀔 때까지 반복해.

**주의할 점 두 가지:**

- EM은 **국소 최적해(local optimum)**에 빠질 수 있어. 초기값을 여러 번 바꿔서 돌리고 가능도가 가장 높은 결과를 써야 해.
- **전체 표본으로 파라미터를 추정한 뒤 필터링 확률을 쓰는 것도 look-ahead야.** 필터링 확률 자체는 과거만 보지만, 그 계산에 들어간 $\mu_k$, $\sigma_k$, $p_{ij}$는 미래 데이터로 추정된 값이니까. 엄밀한 백테스트는 **확장·롤링 윈도우로 파라미터를 다시 추정**해야 해. 이것도 16번에서 다룰게.

---

## 7단계: 예측

오늘의 필터링 확률을 전이행렬로 한 번 굴리면 내일 레짐 확률이 나와.

$$
P(S_{t+1} = j \mid r_{1:t}) = \sum_i \xi_t(i)\, p_{ij}
$$

내일 수익률의 분포는 두 정규분포를 이 확률로 섞은 **혼합분포(mixture)**가 돼. 이 혼합분포의 평균과 분산은 다음과 같아.

$$
E[r_{t+1}] = \sum_k \pi_k\,\mu_k, \qquad \text{Var}(r_{t+1}) = \sum_k \pi_k(\sigma_k^2 + \mu_k^2) - \Big(\sum_k \pi_k\,\mu_k\Big)^2
$$

여기서 $\pi_k$는 예측된 레짐 확률이야.

**모멘트와의 연결:** 각 레짐은 정규분포(왜도 0, 첨도 3)인데, 이걸 **섞으면** 왜도와 첨도가 생겨. 평균이 낮은 혼란장이 섞이면 음의 왜도가, 변동성이 큰 분포가 가끔 섞이면 두꺼운 꼬리(첨도 > 3)가 나타나. HMM이 주식 수익률의 fat tail을 자연스럽게 설명하는 이유야.

---

## 8단계: 네 프로젝트에 쓸 때의 함정

**HMM은 사실상 "변동성 레짐"을 찾는다.** 이건 꼭 알아둬야 할 부분이야. 수익률 데이터에서 레짐별 **평균** 차이(예: +0.05% vs −0.10%)는 노이즈에 묻혀서 거의 식별이 안 되고, **분산** 차이(0.8% vs 2.5%)가 레짐 구분을 거의 전부 결정해. 그래서 "강세장/약세장"을 찾으려고 HMM을 돌려도 실제로 나오는 건 "저변동/고변동 레짐"인 경우가 대부분이야. 수익률만으로 추정한 레짐별 평균은 매우 불안정하다는 점을 전제로 해석해야 해.

**라벨 스위칭.** EM은 "레짐 1"과 "레짐 2"의 이름을 마음대로 붙여. 윈도우를 바꿔 다시 추정하면 번호가 뒤바뀔 수 있어. 예를 들어 "변동성이 작은 쪽을 항상 레짐 1로" 같은 규칙으로 매번 재정렬해야 해.

**레짐 개수 K.** BIC 같은 정보기준으로 고를 수 있지만, 레짐을 늘리면 해석이 급격히 어려워지고 과적합 위험이 커져. 2~3개가 일반적이야.

**잦은 전환과 가짜 경보.** 4단계에서 봤듯 극단적인 하루에 민감하게 반응해. 일별 데이터에서는 레짐이 너무 자주 바뀌는 경향이 있고, 이게 10번 JM의 출발점이야.

**방출분포 가정.** 각 레짐 안에서도 수익률은 정규분포보다 꼬리가 두꺼워. 정규분포를 가정하면 큰 움직임을 "레짐 전환"으로 과잉 해석할 수 있어. t분포 방출을 쓰는 확장도 있어.

**특징 변수(feature) 선택.** 수익률 하나만 넣지 않고, **수익률과 로그 RV를 함께** 넣는 다변량 HMM을 쓰면 레짐 식별이 훨씬 안정적이야. 네 경우엔 10분봉으로 만든 RV나 앞에서 배운 $RS^-/RV$를 특징 변수로 쓸 수 있어. 다만 처음 대화에서 말한 순환성 문제는 조심해야 해. **레짐을 정의하는 변수와, 레짐 안에서 예측에 쓰는 변수가 같으면 결과가 자기 자신을 설명하는 꼴**이 돼.

---

## 계산 코드

`hmmlearn` 라이브러리로 쉽게 추정할 수 있어. **다만 `predict_proba`가 주는 확률은 스무딩 확률(전체 표본 사용)이야.** 백테스트에는 절대 쓰면 안 되고, 필터링 확률은 직접 계산해야 해.

```python
import numpy as np
from hmmlearn.hmm import GaussianHMM
from scipy.stats import norm

X = returns.values.reshape(-1, 1)          # 일별 수익률 (%)

# 1) 추정: 초기값 여러 번 → 가능도 최대인 모형 선택
best, best_ll = None, -np.inf
for seed in range(10):
    m = GaussianHMM(n_components=2, covariance_type='full',
                    n_iter=500, random_state=seed).fit(X)
    ll = m.score(X)
    if ll > best_ll:
        best, best_ll = m, ll

# 라벨 정렬: 변동성 작은 쪽을 레짐 0으로
sig = np.sqrt(best.covars_.flatten())
order = np.argsort(sig)
mu, sig = best.means_.flatten()[order], sig[order]
P = best.transmat_[np.ix_(order, order)]

# 2) 필터링 확률 직접 계산 (각 시점에서 과거만 사용)
def hamilton_filter(r, mu, sig, P, xi0):
    T, K = len(r), len(mu)
    xi = np.zeros((T, K))
    prev = xi0
    for t in range(T):
        pred = prev @ P                      # ① 예측
        lik = norm.pdf(r[t], mu, sig)        # 가능도
        post = pred * lik
        xi[t] = post / post.sum()            # ② 갱신
        prev = xi[t]
    return xi

pi0 = np.array([P[1, 0], P[0, 1]]) / (P[0, 1] + P[1, 0])   # 정상분포로 초기화
xi_filtered = hamilton_filter(returns.values, mu, sig, P, pi0)

# 주의: 여기서도 mu, sig, P를 전체 표본으로 추정했으므로 엄밀한 표본 외 분석이 아님
#       → 롤링/확장 윈도우 재추정은 16번에서
```

---

## 네 프로젝트와의 연결

처음 대화에서 "시장 수준에서 레짐을 정의하라"고 했지. 구체적으로는 이렇게 쓰면 돼.

1. KOSPI 지수의 일별 수익률 + 로그 RV로 2-레짐 HMM을 추정한다.
2. 각 날짜의 **필터링** 확률로 "고변동 레짐일 확률"을 얻는다.
3. 이 확률을 앞에서 배운 모형들의 **상호작용항**으로 넣는다. 예를 들어 HAR에서 $\beta^-$가 고변동 레짐에서 더 커지는지, RSJ의 수익률 예측력이 레짐에 따라 달라지는지 보는 식이야.

이렇게 하면 레짐 정의(시장 지수)와 예측 대상(개별 종목)이 분리돼서 순환성 문제도 피할 수 있어.

---

**한 줄 요약:** HMM은 관측할 수 없는 레짐이 전이행렬에 따라 바뀌고, 레짐마다 수익률 분포가 다르다고 가정하는 모형이야. 매일 "예측(전이행렬) → 갱신(베이즈 정리)"을 반복해서 오늘의 레짐 확률을 추론해. 핵심 함정은 둘이야. **스무딩 확률은 미래를 본 것**이라 백테스트에 쓰면 안 되고, 수익률로 찾은 레짐은 대부분 **변동성 레짐**이야.

다음 9번 **Markov switching**은 HMM과 거의 같은 뿌리인데, 방출분포 대신 **회귀식의 계수 자체가 레짐에 따라 바뀌는** 구조야. HMM을 이해했으면 금방 이해될 거야. 준비되면 말해줘.